In [1]:
import zipfile
import os

# Path to the ZIP file
zip_path = "MP_FIND_DIPPE.zip"

# Folder where files will be extracted
extract_to = "MP_FIND_DIPPE"

# Extract the ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"✅ ZIP extracted to: {extract_to}")

✅ ZIP extracted to: MP_FIND_DIPPE


In [2]:
import os
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# === CONFIG ===
BASE_DIR = "MP_FIND_DIPPE"  # Path to your extracted BOQ zip
OUTPUT_FILE = "MP_FIND_DIPPE.xlsx"

# === HELPER FUNCTIONS ===
def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_class(text):
    text = str(text).lower()
    if 'k-9' in text or 'k9' in text:
        return 'DI-K9'
    elif 'k-7' in text:
        return 'DI-K7'
    elif 'm.s' in text or 'ms' in text:
        return 'MS'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'pvc' in text:
        return 'PVC'
    return None

def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

def find_column(columns, keywords):
    for col in columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

# === MAIN PROCESSING ===
results = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.lower().endswith(('.xls', '.xlsx')) and '_filtered' not in file.lower() and not file.startswith("~$"):
            file_path = os.path.join(root, file)

            # Extract BOQ number from filename (e.g., BOQ_503616.xls → 503616)
            boq_match = re.search(r'boq[_-]?(\d+)', file.lower())
            boq_no = boq_match.group(1) if boq_match else "Unknown"

            try:
                df_raw = pd.read_excel(file_path, header=None)
            except Exception as e:
                print(f"❌ Cannot read: {file_path} - {e}")
                continue

            # Identify header
            header_index = None
            for i in range(min(20, len(df_raw))):
                row = df_raw.iloc[i].astype(str).str.lower()
                if any("desc" in cell or "item" in cell for cell in row) and any("unit" in cell for cell in row):
                    header_index = i
                    break

            if header_index is None:
                continue

            try:
                df = pd.read_excel(file_path, header=header_index)
            except:
                continue

            desc_col = find_column(df.columns, ['desc', 'item'])
            unit_col = find_column(df.columns, ['unit'])
            qty_col = find_column(df.columns, ['qty', 'quantity'])
            rate_col = find_column(df.columns, ['rate', 'amount', 'estimate'])

            if not desc_col or not unit_col:
                continue

            df["Item Description"] = df[desc_col].astype(str)
            df["class"] = df["Item Description"].apply(extract_class)
            df["DIA"] = df["Item Description"].apply(extract_dia)
            df["Units"] = df[unit_col].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
            df["Quantity"] = pd.to_numeric(df[qty_col], errors='coerce') if qty_col else None
            df["Estimate Rate"] = pd.to_numeric(df[rate_col], errors='coerce') if rate_col else None

            # Filter pipe-related rows and "mtr" unit only
            df_pipe = df[df["Item Description"].str.contains("pipe|di|k-9|k-7|m.s|hdpe|pvc", case=False, na=False)]
            df_pipe = df_pipe[df_pipe["Units"] == "mtr"]

            if df_pipe.empty:
                results.append({
                    "BOQ_NO": boq_no, "Item Description": "", "DI-K9": "", "DI-K7": "", "MS": "",
                    "HDPE": "", "PVC": "", "DIA": "", "Estimate Rate": "", "Units": "", "Quantity": ""
                })
            else:
                for _, row in df_pipe.iterrows():
                    row_data = {
                        "BOQ_NO": boq_no,
                        "Item Description": clean_illegal_chars(row["Item Description"]),
                        "DI-K9": "Yes" if row["class"] == "DI-K9" else "",
                        "DI-K7": "Yes" if row["class"] == "DI-K7" else "",
                        "MS": "Yes" if row["class"] == "MS" else "",
                        "HDPE": "Yes" if row["class"] == "HDPE" else "",
                        "PVC": "Yes" if row["class"] == "PVC" else "",
                        "DIA": row["DIA"],
                        "Estimate Rate": row["Estimate Rate"],
                        "Units": row["Units"],
                        "Quantity": row["Quantity"]
                    }
                    results.append(row_data)

# === EXPORT ===
df_summary = pd.DataFrame(results)
df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
df_summary.to_excel(OUTPUT_FILE, index=False)
print(f"✅ Output saved to: {OUTPUT_FILE}")

✅ Output saved to: MP_FIND_DIPPE.xlsx
